# 01 — Data Cleaning

Pipeline:
1. Load raw `results_linux.csv`
2. Detect & remove outliers (IQR × 1.5, per language × benchmark group)
3. Convert units to human-readable values
4. Aggregate to mean per (language, benchmark)
5. Export `results_clean.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

RAW_CSV  = Path('../results/results_linux.csv')
OUT_CSV  = Path('../results/results_clean.csv')
FIGS_DIR = Path('../results/figs')
FIGS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load raw data

In [ ]:
df = pd.read_csv(RAW_CSV)
print(f'Shape: {df.shape}')
df.head(3)

## 2. Outlier detection & removal (IQR per group)

In [ ]:
# Numeric metric columns only (exclude identifiers)
ID_COLS     = ['run_id', 'measured_at', 'language', 'benchmark']
METRIC_COLS = [c for c in df.columns if c not in ID_COLS]

def iqr_mask(group: pd.DataFrame, cols: list[str], factor: float = 1.5) -> pd.Series:
    """Return a boolean mask — True means the row is an outlier on ≥1 metric."""
    outlier = pd.Series(False, index=group.index)
    for col in cols:
        q1, q3 = group[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:          # constant column in this group — skip
            continue
        lo, hi = q1 - factor * iqr, q3 + factor * iqr
        outlier |= (group[col] < lo) | (group[col] > hi)
    return outlier

outlier_flags = (
    df.groupby(['language', 'benchmark'], group_keys=False)
      .apply(lambda g: iqr_mask(g, METRIC_COLS))
)

n_outliers = outlier_flags.sum()
print(f'Outlier runs flagged : {n_outliers} / {len(df)}  ({n_outliers/len(df)*100:.1f}%)')

df_clean = df[~outlier_flags].copy()
print(f'Rows after removal   : {len(df_clean)}')

In [ ]:
# Per-group count after cleaning — flag any group with fewer than 5 clean runs
counts = df_clean.groupby(['language', 'benchmark']).size().rename('n_runs')
low = counts[counts < 5]
if len(low):
    print('⚠ Groups with < 5 clean runs:')
    print(low.to_string())
else:
    print('All groups have ≥ 5 clean runs.')
counts.unstack('benchmark').fillna(0).astype(int)

In [ ]:
# Visual: distribution of clean run counts per group
fig, ax = plt.subplots(figsize=(5, 3))
counts.value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Runs kept per group')
ax.set_ylabel('Number of groups')
ax.set_title('Clean run counts per (language × benchmark)')
plt.tight_layout()
plt.savefig(FIGS_DIR / 'outlier_counts.pdf', bbox_inches='tight')
plt.show()

## 3. Unit conversions

In [ ]:
# Conversion rules keyed by column-name suffix.
# Each entry: (divisor, new_unit_label)
CONVERSIONS = {
    '-ug' : (1e6,    'g'),        # micro-grams  → grams
    '-uj' : (1e6,    'J'),        # micro-Joules → Joules
    '-mw' : (1e3,    'W'),        # milli-Watts  → Watts
    '-us' : (1e6,    's'),        # micro-seconds → seconds
    '-bytes': (1e6,  'MB'),       # bytes → MB  (disk/net totals)
}

rename_map = {}
for col in METRIC_COLS:
    for suffix, (divisor, unit) in CONVERSIONS.items():
        if col.endswith(suffix):
            base = col[: -len(suffix)]
            new_col = f'{base}-{unit.lower()}'
            df_clean[new_col] = df_clean[col] / divisor
            rename_map[col] = new_col
            break

# Drop the original raw-unit columns
df_clean = df_clean.drop(columns=list(rename_map.keys()))

METRIC_COLS_CONV = [c for c in df_clean.columns if c not in ID_COLS]
print('Converted columns:')
for old, new in rename_map.items():
    print(f'  {old}  →  {new}')

## 4. Aggregate to mean per (language, benchmark)

In [ ]:
df_mean = (
    df_clean
      .groupby(['language', 'benchmark'])[METRIC_COLS_CONV]
      .mean()
      .reset_index()
)
print(f'Aggregated shape: {df_mean.shape}')
df_mean.head()

## 5. Export

In [ ]:
df_mean.to_csv(OUT_CSV, index=False)
print(f'Saved → {OUT_CSV}')

# Also save the per-run cleaned frame (before aggregation) for reference
df_clean.to_csv(OUT_CSV.with_name('results_clean_runs.csv'), index=False)
print(f'Saved → {OUT_CSV.with_name("results_clean_runs.csv")}')